# Image Embedding Extraction with ResNet-18 (PyTorch)

This notebook explains **how an image is converted into numbers** using a pretrained ResNet‑18 model.

An *image embedding* is a list of numbers that summarizes what an image looks like.
These numbers can be used later for machine learning, comparison, or analysis.

This tutorial shows:
- How a neural network extracts visual features
- How features change from early to deep layers
- How the final 512‑D image embedding is created
- How embeddings are saved as a table (CSV)


## Import required libraries

We load the libraries needed to:
- run the neural network (PyTorch)
- process images
- visualize results
- store data in tables


In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from torchvision import models, transforms
from PIL import Image


## Choose computation device

We explicitly choose **CPU** so the notebook works reliably
on Binder and other online platforms.


In [ ]:
device = torch.device('cpu')
print('Using device:', device)


## Load ResNet‑18 in embedding mode

ResNet‑18 is a pretrained image recognition model.

Normally it ends with a classifier, but we **remove that layer** so the model
outputs a numerical description (embedding) instead of a class label.


In [ ]:
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
model.fc = nn.Identity()
model.eval()
print('ResNet-18 loaded in embedding mode')


## Capture feature maps from each layer

As the image passes through the network, each layer looks at it differently:
- early layers detect edges
- middle layers detect textures
- deep layers detect structures

We use hooks to record what each layer sees.


In [ ]:
feature_maps = {}

def save_hook(name):
    def hook(module, inp, out):
        feature_maps[name] = out.detach().cpu()
    return hook

model.conv1.register_forward_hook(save_hook('conv1'))
model.layer1.register_forward_hook(save_hook('layer1'))
model.layer2.register_forward_hook(save_hook('layer2'))
model.layer3.register_forward_hook(save_hook('layer3'))
model.layer4.register_forward_hook(save_hook('layer4'))

print('Feature-map hooks registered')


## Image preprocessing

The image must be prepared in the same way the model was trained.
This includes resizing, cropping, converting to numbers, and normalization.


In [ ]:
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(256),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])


## Load and display the image

We load the image and display it so we know what we are analyzing.


In [ ]:
img = Image.open('data/25DIV3FAY_1002.jpg').convert('RGB')
x = transform(img).unsqueeze(0)

plt.figure(figsize=(4,4))
plt.imshow(img)
plt.axis('off')
plt.title('Input Image')
plt.show()


## Forward pass and embedding extraction

The image now goes through the network.
The result is a **512‑dimensional embedding**.


In [ ]:
with torch.no_grad():
    embedding = model(x).squeeze().numpy()

print('Embedding shape:', embedding.shape)


## Select representative feature maps

Each layer has many feature maps. We show only a few examples to keep
the visualization readable.


In [ ]:
def representative_maps(fmap, n=3):
    fmap = fmap.squeeze(0)
    C = fmap.shape[0]
    idxs = np.linspace(0, C - 1, n, dtype=int)
    maps = []
    for i in idxs:
        fm = fmap[i]
        fm = (fm - fm.min()) / (fm.max() - fm.min() + 1e-6)
        maps.append(fm)
    return maps


## Layer‑wise visualization

This diagram shows how the image representation changes as it moves
through the network layers, ending in a numerical embedding.


In [ ]:
fig, ax = plt.subplots(figsize=(20,6))
ax.axis('off')

x_pos = 2
y_base = 1

ax.imshow(img.resize((128,128)), extent=(0,1,2,3))
ax.text(0.5,1.6,'Input\n256\u00d7256\u00d73',ha='center')

for name in ['conv1','layer1','layer2','layer3','layer4']:
    fmap = feature_maps[name]
    C,H,W = fmap.shape[1:]
    reps = representative_maps(fmap, n=3)
    for i,fm in enumerate(reps):
        ax.imshow(fm, extent=(x_pos,x_pos+0.8,y_base+i*0.9,y_base+i*0.9+0.8), cmap='viridis')
    ax.text(x_pos+0.4,y_base-0.3,f'{name}\n{C}\u00d7{H}\u00d7{W}',ha='center',fontsize=8)
    x_pos += 1.1

ax.bar(np.linspace(x_pos,x_pos+2,512), embedding, width=0.003, color='black')
ax.text(x_pos+1,1.5,'Embedding\n512-D',ha='center')

plt.title('ResNet-18: Layer-wise Feature Maps and Image Embedding')
plt.tight_layout()
plt.show()


## Store embedding as a table

We convert the embedding into a table so it can be saved and reused.


In [ ]:
df_embedding = pd.DataFrame([
    embedding
], columns=[f'emb_{i}' for i in range(len(embedding))])
df_embedding.insert(0, 'image_id', 'example_image.jpg')
df_embedding


## Save embedding to CSV

This allows the embedding to be downloaded and reused without rerunning the model.


In [ ]:
df_embedding.to_csv('image_embedding.csv', index=False)
print('Embedding saved to image_embedding.csv')
